# NB04: Cohort labels and gene signatures

Builds the patient-level label table for downstream training. Combines:
- Patient registry from the WSI manifest produced by NB02
- Three RNA-seq gene signatures via UCSC Xena Toil (IFNG6, ANGIO, HRD_expr)
- Stratified 80/10/10 train/val/test split by cancer type

**Manuscript reference**:
- Gene panels (Supp Table S7): IFNG6 (6 genes), ANGIO (21 genes), HRR_CORE (23 genes)
- Splits (Methods, Supp S5): 6,465 train / 811 val / 833 test, stratified by cancer type, seed=42
- Cross-signature correlations (Supp Table S9): consumed by NB10

**Required env**: `WORKSPACE`, `WSI_ROOT` (or a prior NB02 tile run).
**Outputs**: `labels/labels.parquet`, `labels/labels.csv` (HRD column added by NB05).

In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
WSI_ROOT = Path(os.environ["WSI_ROOT"]).resolve() if os.environ.get("WSI_ROOT") else None
LABELS_DIR = WORKSPACE / "labels"
RUNS_DIR = WORKSPACE / "runs"
RESULTS_DIR = WORKSPACE / "results" / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked seed (Supp S5)
SEED = 42

# UCSC Xena Toil RNA-seq dataset (log2(TPM+0.001))
XENA_HUBS = [
    "https://toil.xenahubs.net",
    "https://gdc.xenahubs.net",
    "https://pancanatlas.xenahubs.net",
]
XENA_RNA_DATASET = "TcgaTargetGtex_rsem_gene_tpm"
print(f"WORKSPACE: {WORKSPACE}")
print(f"SEED     : {SEED}")


In [ ]:
# gene panels (Supplementary Table S7)
IFNG6 = ["IFNG", "STAT1", "IDO1", "CXCL9", "CXCL10", "HLA-DRA"]
ANGIO = [
    "VEGFA", "KDR", "FLT1", "ANGPT2", "TEK", "ENG", "PECAM1", "VWF",
    "COL4A1", "COL4A2", "MMP2", "MMP9", "HIF1A", "PDGFB", "PDGFRB",
    "ITGAV", "ITGB3", "ICAM1", "SELE", "PGF", "ELN",
]
HRR_CORE = [
    "BRCA1", "BRCA2", "PALB2", "RAD51", "RAD51C", "RAD51D", "BARD1",
    "BRIP1", "ATM", "ATR", "CHEK1", "CHEK2", "XRCC2", "XRCC3",
    "MRE11", "RAD50", "NBN", "FANCA", "FANCD2", "FANCM", "RPA1",
    "RFC2", "BLM",
]
GENE_UNION = sorted(set(IFNG6 + ANGIO + HRR_CORE))
print(f"gene panels: IFNG6={len(IFNG6)}, ANGIO={len(ANGIO)}, HRR_CORE={len(HRR_CORE)}")
print(f"unique genes total: {len(GENE_UNION)}")


In [ ]:
# build registry: prefer NB02 summary, else scan WSI_ROOT directly
def patient_id_from_name(name):
    m = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", name, re.I)
    return m.group(1).upper() if m else None

def cancer_from_path(p, root):
    try:
        rel = Path(p).relative_to(root)
        return rel.parts[0].upper() if len(rel.parts) > 1 else "UNK"
    except Exception:
        return "UNK"

def latest_tile_summary():
    cands = sorted(RUNS_DIR.glob("tile_*/summary.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0] if cands else None

reg_path = latest_tile_summary()
if reg_path is not None:
    summary = pd.read_csv(reg_path)
    print(f"using NB02 tile-run summary: {reg_path}")
    rows = []
    for _, row in summary.iterrows():
        sid = row["slide_id"]
        path = row.get("path", "")
        rows.append({
            "slide_id": sid,
            "patient": patient_id_from_name(sid),
            "cancer": cancer_from_path(path, WSI_ROOT) if WSI_ROOT and path else "UNK",
        })
    slides_df = pd.DataFrame(rows)
elif WSI_ROOT is not None:
    print(f"scanning {WSI_ROOT} directly ...")
    rows = []
    exts = (".svs", ".tif", ".tiff", ".ndpi", ".mrxs", ".scn", ".bif")
    for p in WSI_ROOT.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            rows.append({
                "slide_id": p.stem,
                "patient": patient_id_from_name(p.name),
                "cancer": cancer_from_path(p, WSI_ROOT),
            })
    slides_df = pd.DataFrame(rows)
else:
    raise SystemExit("no NB02 summary and no WSI_ROOT set; cannot build registry")

slides_df = slides_df.dropna(subset=["patient"]).drop_duplicates(subset=["slide_id"])
print(f"slides: {len(slides_df):,} | unique patients: {slides_df['patient'].nunique():,}")
print(f"cancer types: {slides_df['cancer'].nunique()}")


In [ ]:
# fetch RNA via Xena (xenaPython auto-installed if missing)
def install_xena():
    try:
        import xenaPython  # noqa
    except Exception:
        import subprocess
        subprocess.check_call(["pip", "install", "--quiet", "xenaPython"])

install_xena()
import xenaPython as xena

def find_xena_hub():
    for h in XENA_HUBS:
        try:
            _ = xena.dataset_samples(h, XENA_RNA_DATASET, 5)
            return h
        except Exception:
            continue
    return None

hub = find_xena_hub()
if hub is None:
    raise SystemExit(f"none of the Xena hubs are reachable for {XENA_RNA_DATASET}")
print(f"xena hub: {hub}")

samples = xena.dataset_samples(hub, XENA_RNA_DATASET, None)
samples = [s for s in samples if s.startswith("TCGA-")]
print(f"TCGA samples on hub: {len(samples):,}")


In [ ]:
def fetch_genes(hub, samples, genes):
    out = {}
    for g in genes:
        try:
            _, vals = xena.dataset_probe_values(hub, XENA_RNA_DATASET, samples, [g])
            out[g] = pd.Series(vals[0], index=samples, dtype="float32")
        except Exception:
            out[g] = pd.Series([np.nan] * len(samples), index=samples, dtype="float32")
    df = pd.DataFrame(out)
    df.index.name = "sample"
    return df

print(f"fetching {len(GENE_UNION)} genes ...")
rna = fetch_genes(hub, samples, GENE_UNION)
print(f"RNA matrix: {rna.shape}")


In [ ]:
# collapse to 12-char patient barcode, prefer primary tumor sample (-01)
def collapse_to_patient(mat):
    df = mat.copy()
    df["patient"] = df.index.str.slice(0, 12)
    df["is_primary"] = df.index.str.contains(r"-01[A-Z]?$", regex=True)
    df = df.sort_values("is_primary", ascending=False).drop(columns=["is_primary"])
    return df.groupby("patient").first()

pt_expr = collapse_to_patient(rna)
print(f"per-patient expression: {pt_expr.shape}")


In [ ]:
# z-score per gene across patients, then average within each panel
def zscore_columns(df):
    return (df - df.mean(axis=0, skipna=True)) / (df.std(axis=0, ddof=0, skipna=True) + 1e-8)

scores = pd.DataFrame(index=pt_expr.index)
if not pt_expr.empty:
    expr_z = zscore_columns(pt_expr)

    avail_ifng = [g for g in IFNG6 if g in expr_z.columns]
    scores["IFNG6"] = expr_z[avail_ifng].mean(axis=1) if avail_ifng else np.nan

    avail_ang = [g for g in ANGIO if g in expr_z.columns]
    scores["ANGIO"] = expr_z[avail_ang].mean(axis=1) if avail_ang else np.nan

    # HRD_expr = -mean z of HRR (higher = more deficient)
    avail_hrr = [g for g in HRR_CORE if g in expr_z.columns]
    scores["HRD_expr"] = -expr_z[avail_hrr].mean(axis=1) if avail_hrr else np.nan

print("signature non-null counts:")
for c in ["IFNG6", "ANGIO", "HRD_expr"]:
    if c in scores.columns:
        print(f"  {c}: {scores[c].notna().sum():,}")


In [ ]:
# patient-level table: cancer + signatures + slide list
sl_by_pt = slides_df.groupby("patient")["slide_id"].apply(list).rename("slides")
pt_cancer = slides_df.groupby("patient")["cancer"].agg(lambda x: x.mode().iat[0] if len(x) > 0 else "UNK")

labels = pd.DataFrame(index=sorted(set(scores.index).intersection(sl_by_pt.index)))
labels = labels.join(scores, how="left")
labels = labels.join(sl_by_pt, how="left")
labels = labels.join(pt_cancer.rename("cancer"), how="left")
labels.reset_index(inplace=True)
labels = labels.rename(columns={"index": "patient"})

print(f"patients with WSIs + RNA: {len(labels):,}")


In [ ]:
# stratified 80/10/10 split per cancer, seed=42 (manuscript Methods + Supp S5)
rng = np.random.RandomState(SEED)
labels["_rand"] = rng.rand(len(labels))
labels["split"] = (
    labels.groupby("cancer")["_rand"]
    .transform(lambda s: pd.qcut(
        s.rank(method="first"),
        q=[0, 0.8, 0.9, 1.0],
        labels=["train", "val", "test"],
    ))
)
labels = labels.drop(columns=["_rand"])
labels["split"] = labels["split"].astype(str)

split_counts = labels["split"].value_counts().sort_index()
print(split_counts)
print()
print("manuscript ref: 6,465 train / 811 val / 833 test (Supp S5)")


In [ ]:
# write outputs (HRD numeric column added by NB05)
out_pq = LABELS_DIR / "labels.parquet"
out_csv = LABELS_DIR / "labels.csv"
labels.to_parquet(out_pq, index=False)
labels.to_csv(out_csv, index=False)

diag = {
    "n_patients": int(len(labels)),
    "n_cancers": int(labels["cancer"].nunique()),
    "splits": split_counts.to_dict(),
    "signatures": {
        "IFNG6": int(labels["IFNG6"].notna().sum()),
        "ANGIO": int(labels["ANGIO"].notna().sum()),
        "HRD_expr": int(labels["HRD_expr"].notna().sum()),
    },
}
(RESULTS_DIR / "labels_summary.json").write_text(json.dumps(diag, indent=2))
print(json.dumps(diag, indent=2))
print(f"\nwritten: {out_pq}")
